In [0]:
%run /Workspace/Users/vamshialwayskush@gmail.com/Brazilian_ECommerce/ImportPackages

In [0]:
from datetime import timedelta, datetime
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window as W
from pyspark.storagelevel import StorageLevel
from delta.tables import *
from functools import reduce

import os #for adding filename to dataframe
import re #regular expressions
import time

In [0]:
%sql
create schema if not exists brazilian_ecommerce.raw;
create schema if not exists brazilian_ecommerce.silver;
create schema if not exists brazilian_ecommerce.gold;
create schema if not exists brazilian_ecommerce.platinum;

In [0]:
# List all files in the DBFS folder
files = dbutils.fs.ls("/Volumes/brazilian_ecommerce/raw/raw_datasets/")
# print(files)

# Filter for CSV files and extract only the file names
csv_file_names = [f.name for f in files if f.name.endswith(".csv")]

# Display the file names
print(csv_file_names)

success = True

In [0]:
# Get current UTC time to be used for entire batch of files to be ingested
dtUTC = datetime.utcnow()
BATCH_TIME = dtUTC.isoformat() # UTC timestamp for type 2 column
BATCH_TIME_STR = dtUTC.strftime('%Y-%m-%dT%H:%M:%S.%fZ') # UTC timestamp for MERGE commands

catalog_name = "brazilian_ecommerce"
target_schema = "raw"
file_path  = "/Volumes/brazilian_ecommerce/raw/raw_datasets/"


In [0]:
success = True

if success:
    try:
        for file in csv_file_names:
            file_name = file
            # file path and file name
            file_path_name = f"{file_path}{file_name}"
            # raw table name
            target_table_name = file_name.replace(".csv", "")

            if target_table_name.endswith("_dataset"):
                target_table_name = target_table_name.replace("_dataset", "")
            else:
                target_table_name
            
            # Create a view to use in SQL statements      
            temp_vw_name_raw = 'tmpvw_raw_' + target_table_name

            print("file path and name: " + file_path_name)
            print("temporary view name: " + temp_vw_name_raw)
            print("target table name: " + target_table_name)

            target_file_location_name = f"{catalog_name}.{target_schema}.{target_table_name}"
            print(f"Target table name: {target_file_location_name}")
            
            print(f"----- Reading {file_name} from {file_path} -----")

            df = spark.read.format("csv")\
                .option("inferSchema", True)\
                .option("header", True)\
                .load(f"{file_path_name}")

            #  .option("mode", "FAILFAST")\
                
            # Drop Duplicates
            df = df.dropDuplicates()

            df.createOrReplaceTempView(temp_vw_name_raw)

            print(f"----- Creating temporary view {temp_vw_name_raw} -----")
            sdf = spark.table(temp_vw_name_raw) # 11s 

            columns = [f"`{column}`" for column in sdf.columns]

            print(f"----- Adding metadata columns to temporary view {temp_vw_name_raw} -----")

            # FIRST: Add sha2 hash over all source columns for SCD Type 2
            sdf = sdf.withColumn("dl_rowhash", sha2(concat_ws("||", *columns), 256))
                
            # Add an Active flag, setting it to True
            sdf = sdf.withColumn("dl_iscurrent", lit(True)) #Add column indicating the pDeltaObjectName #2.24m

            # Add an IsDeleted flag, setting it to False
            sdf = sdf.withColumn("dl_isdeleted", lit(False))
            
            # Add Record dates for SCD Type 2
            sdf = sdf.withColumn('dl_recordstartdateutc', current_timestamp()) #add timestamp when processed
            sdf = sdf.withColumn("dl_recordenddateutc", to_timestamp(lit(None), "yyyy-MM-dd HH:mm:ss")) #Add column indicating the type 2 end date

            # Create a temp table from Dataframe to use in MERGE statement into Delta Lake
            vw_name_enriched = 'tmpvw_enriched_' + temp_vw_name_raw

            sdf.createOrReplaceTempView(vw_name_enriched)

            sdf1 = spark.table(vw_name_enriched)
            print(f"----- Created temporary view {vw_name_enriched} -----")

            print(f"----- Processing {file_name} into {target_file_location_name} -----")
            sdf1.write.format("delta").mode("append").saveAsTable(f"{target_file_location_name}")
            print(f"----- Processed {file_name} into {target_file_location_name} -----")


    except Exception as e:
        print(f"Error: {e}")
        success = False
